# Calculate Maritime Distances Using Searoute

This notebook uses the `searoute` library to compute actual maritime shipping distances for missing port-to-port routes.

**Methodology:**
- Uses open-source maritime routing (searoute library)
- Accounts for shipping lanes, canals (Suez, Panama), and land avoidance
- Returns distances in nautical miles
- Standard approach in academic maritime research

**Output:** Calculated distances will be appended to `Port Distances.csv`

In [1]:
# Cell 1: Install required libraries
# Run this cell first - it will install searoute if not already installed
%pip install searoute pandas -q

print("✓ Libraries installed/verified")

Note: you may need to restart the kernel to use updated packages.
✓ Libraries installed/verified


In [2]:
# Cell 2: Import libraries and load data
import pandas as pd
import searoute as sr
from pathlib import Path
import time

base = Path("data")

# Load missing routes
print("Loading missing routes...")
missing_routes = pd.read_csv("missing_port_distances.csv")
print(f"Found {len(missing_routes)} missing routes")

# Load port locations
print("Loading port locations...")
port_locations = pd.read_csv(base / "port_locations.csv")
print(f"Found {len(port_locations)} ports with coordinates")

print("\n✓ Data loaded successfully!")

Loading missing routes...
Found 63 missing routes
Loading port locations...
Found 28 ports with coordinates

✓ Data loaded successfully!


In [3]:
# Cell 3: Create port alias mapping and coordinate lookup
# Handle port name variations (e.g., "GWANGYANG LNG TERMINAL" vs "GWANGYANG")

port_aliases = {
    # Add any port name variations here if needed
    # Example: 'GWANGYANG LNG TERMINAL': 'GWANGYANG',
}

# Build coordinate lookup dictionary
port_coords = {}
for _, row in port_locations.iterrows():
    port_name = str(row['port_name']).strip()
    lat = row['latitude']
    lon = row['longitude']
    
    # Store coordinates
    port_coords[port_name] = (lon, lat)  # searoute expects (longitude, latitude)
    
    # Also store alias if exists
    if port_name in port_aliases:
        alias = port_aliases[port_name]
        if alias not in port_coords:
            port_coords[alias] = (lon, lat)

print(f"✓ Built coordinate lookup for {len(port_coords)} ports")
print(f"Sample ports: {list(port_coords.keys())[:5]}")

✓ Built coordinate lookup for 28 ports
Sample ports: ['DAMPIER', 'PONTA DA MADEIRA', 'SALDANHA BAY', 'TABONEO', 'VANCOUVER (CANADA)']


In [4]:
# Cell 4: Test searoute with 2-3 sample routes
# This verifies the library works before calculating all 63 routes

print("Testing searoute with sample routes...")
print("=" * 60)

# Get first 3 routes as test
test_routes = missing_routes.head(3)

for idx, row in test_routes.iterrows():
    port_from = str(row['PORT_NAME_FROM']).strip()
    port_to = str(row['PORT_NAME_TO']).strip()
    
    # Check if ports exist
    if port_from not in port_coords:
        print(f"❌ {port_from} not found in port_locations.csv")
        continue
    if port_to not in port_coords:
        print(f"❌ {port_to} not found in port_locations.csv")
        continue
    
    origin = port_coords[port_from]
    destination = port_coords[port_to]
    
    try:
        # Calculate route
        route = sr.searoute(origin, destination)
        distance_nm = route.properties["length"]
        
        print(f"✓ {port_from} → {port_to}")
        print(f"  Distance: {distance_nm:.2f} nm")
        print(f"  Route coordinates: {len(route.geometry['coordinates'])} points")
        print()
    except Exception as e:
        print(f"❌ Error calculating {port_from} → {port_to}: {e}")
        print()

print("=" * 60)
print("✓ Test complete. If all tests passed, proceed to Cell 5.")

Testing searoute with sample routes...
✓ QINGDAO → TABONEO
  Distance: 4275.32 nm
  Route coordinates: 22 points

✓ MAP TA PHUT → KAMSAR ANCHORAGE
  Distance: 17650.55 nm
  Route coordinates: 94 points

✓ MAP TA PHUT → PORT HEDLAND
  Distance: 4643.39 nm
  Route coordinates: 16 points

✓ Test complete. If all tests passed, proceed to Cell 5.


In [5]:
# Cell 5: Calculate distances for all missing routes
# This may take 1-5 minutes for 63 routes

print("Calculating maritime distances for all missing routes...")
print(f"Total routes to calculate: {len(missing_routes)}")
print("=" * 60)

calculated_distances = []
errors = []

for idx, row in missing_routes.iterrows():
    port_from = str(row['PORT_NAME_FROM']).strip()
    port_to = str(row['PORT_NAME_TO']).strip()
    
    # Check if ports exist
    if port_from not in port_coords:
        errors.append({
            'PORT_NAME_FROM': port_from,
            'PORT_NAME_TO': port_to,
            'ERROR': f"Port '{port_from}' not found in port_locations.csv"
        })
        print(f"❌ [{idx+1}/{len(missing_routes)}] {port_from} → {port_to}: Missing FROM port")
        continue
        
    if port_to not in port_coords:
        errors.append({
            'PORT_NAME_FROM': port_from,
            'PORT_NAME_TO': port_to,
            'ERROR': f"Port '{port_to}' not found in port_locations.csv"
        })
        print(f"❌ [{idx+1}/{len(missing_routes)}] {port_from} → {port_to}: Missing TO port")
        continue
    
    origin = port_coords[port_from]
    destination = port_coords[port_to]
    
    try:
        # Calculate route using searoute
        route = sr.searoute(origin, destination)
        distance_nm = route.properties["length"]
        
        calculated_distances.append({
            'PORT_NAME_FROM': port_from,
            'PORT_NAME_TO': port_to,
            'DISTANCE': round(distance_nm, 2),
            'METHOD': 'searoute',
            'CONFIDENCE': 'high'
        })
        
        print(f"✓ [{idx+1}/{len(missing_routes)}] {port_from} → {port_to}: {distance_nm:.2f} nm")
        
        # Small delay to avoid overwhelming the library
        time.sleep(0.1)
        
    except Exception as e:
        errors.append({
            'PORT_NAME_FROM': port_from,
            'PORT_NAME_TO': port_to,
            'ERROR': str(e)
        })
        print(f"❌ [{idx+1}/{len(missing_routes)}] {port_from} → {port_to}: {e}")

print("\n" + "=" * 60)
print(f"✓ Calculation complete!")
print(f"  Successfully calculated: {len(calculated_distances)} routes")
print(f"  Errors: {len(errors)} routes")

if errors:
    print("\n⚠ Errors encountered:")
    for err in errors:
        print(f"  - {err['PORT_NAME_FROM']} → {err['PORT_NAME_TO']}: {err['ERROR']}")

Calculating maritime distances for all missing routes...
Total routes to calculate: 63
✓ [1/63] QINGDAO → TABONEO: 4275.32 nm
✓ [2/63] MAP TA PHUT → KAMSAR ANCHORAGE: 17650.55 nm
✓ [3/63] MAP TA PHUT → PORT HEDLAND: 4643.39 nm
✓ [4/63] MAP TA PHUT → ITAGUAI: 17965.98 nm
✓ [5/63] MAP TA PHUT → DAMPIER: 4719.05 nm
✓ [6/63] MAP TA PHUT → PONTA DA MADEIRA: 19330.43 nm
✓ [7/63] MAP TA PHUT → SALDANHA BAY: 11876.79 nm
✓ [8/63] MAP TA PHUT → VANCOUVER (CANADA): 13464.58 nm
✓ [9/63] MAP TA PHUT → TUBARAO: 17804.43 nm
✓ [10/63] GWANGYANG LNG TERMINAL → KAMSAR ANCHORAGE: 20788.80 nm
✓ [11/63] GWANGYANG LNG TERMINAL → ITAGUAI: 20821.15 nm
✓ [12/63] GWANGYANG LNG TERMINAL → DAMPIER: 6532.96 nm
✓ [13/63] GWANGYANG LNG TERMINAL → PONTA DA MADEIRA: 20017.76 nm
✓ [14/63] GWANGYANG LNG TERMINAL → SALDANHA BAY: 14731.96 nm
✓ [15/63] GWANGYANG LNG TERMINAL → TABONEO: 4278.00 nm
✓ [16/63] GWANGYANG LNG TERMINAL → TUBARAO: 20659.59 nm
✓ [17/63] FANGCHENG → KAMSAR ANCHORAGE: 18687.74 nm
✓ [18/63] PARADIP → 

In [6]:
# Cell 6: Save calculated distances to CSV

if calculated_distances:
    results_df = pd.DataFrame(calculated_distances)
    results_df.to_csv('searoute_calculated_distances.csv', index=False)
    print(f"✓ Saved {len(calculated_distances)} calculated distances to: searoute_calculated_distances.csv")
    print("\nSample results:")
    print(results_df.head(10).to_string(index=False))
else:
    print("⚠ No distances calculated. Check errors above.")

if errors:
    errors_df = pd.DataFrame(errors)
    errors_df.to_csv('searoute_errors.csv', index=False)
    print(f"\n⚠ Saved {len(errors)} errors to: searoute_errors.csv")

✓ Saved 63 calculated distances to: searoute_calculated_distances.csv

Sample results:
        PORT_NAME_FROM       PORT_NAME_TO  DISTANCE   METHOD CONFIDENCE
               QINGDAO            TABONEO   4275.32 searoute       high
           MAP TA PHUT   KAMSAR ANCHORAGE  17650.55 searoute       high
           MAP TA PHUT       PORT HEDLAND   4643.39 searoute       high
           MAP TA PHUT            ITAGUAI  17965.98 searoute       high
           MAP TA PHUT            DAMPIER   4719.05 searoute       high
           MAP TA PHUT   PONTA DA MADEIRA  19330.43 searoute       high
           MAP TA PHUT       SALDANHA BAY  11876.79 searoute       high
           MAP TA PHUT VANCOUVER (CANADA)  13464.58 searoute       high
           MAP TA PHUT            TUBARAO  17804.43 searoute       high
GWANGYANG LNG TERMINAL   KAMSAR ANCHORAGE  20788.80 searoute       high


In [7]:
# Cell 7: Append new distances to Port Distances.csv
# This adds both directions (A→B and B→A) to match the format

if calculated_distances:
    # Load existing Port Distances
    port_distances = pd.read_csv(base / "Port Distances.csv")
    print(f"Loaded {len(port_distances)} existing routes from Port Distances.csv")
    
    # Create new rows in the same format
    new_rows = []
    for dist_data in calculated_distances:
        # Forward direction
        new_rows.append({
            'PORT_NAME_FROM': dist_data['PORT_NAME_FROM'],
            'PORT_NAME_TO': dist_data['PORT_NAME_TO'],
            'DISTANCE': dist_data['DISTANCE']
        })
        
        # Reverse direction (same distance)
        new_rows.append({
            'PORT_NAME_FROM': dist_data['PORT_NAME_TO'],
            'PORT_NAME_TO': dist_data['PORT_NAME_FROM'],
            'DISTANCE': dist_data['DISTANCE']
        })
    
    new_df = pd.DataFrame(new_rows)
    
    # Combine with existing
    updated_distances = pd.concat([port_distances, new_df], ignore_index=True)
    
    # Remove duplicates (in case any routes already existed)
    updated_distances = updated_distances.drop_duplicates(
        subset=['PORT_NAME_FROM', 'PORT_NAME_TO'],
        keep='first'
    )
    
    # Save updated file
    updated_distances.to_csv(base / "Port Distances.csv", index=False)
    
    print(f"✓ Added {len(new_rows)} new routes (both directions)")
    print(f"✓ Updated Port Distances.csv now has {len(updated_distances)} total routes")
    print(f"\n⚠ IMPORTANT: Re-run your optimization notebook to use the new distances!")
else:
    print("⚠ No distances to append. Run Cell 5 first.")

Loaded 15535 existing routes from Port Distances.csv
✓ Added 126 new routes (both directions)
✓ Updated Port Distances.csv now has 15661 total routes

⚠ IMPORTANT: Re-run your optimization notebook to use the new distances!


## Next Steps

1. ✅ Distances calculated and saved
2. ✅ Port Distances.csv updated
3. **Re-run your optimization notebook** (`vessel_cargo_optimization.ipynb`)
   - The optimization will automatically load the new distances
   - All 63 missing routes should now be found
   - More vessel-cargo pairs should become feasible

## Methodology Notes for Presentation

When presenting, you can say:

> "Port-to-port distances are computed using the open-source searoute library, which uses OpenStreetMap maritime data to calculate actual shipping routes. This accounts for shipping lanes, avoids landmasses, and considers canal routes (Suez, Panama). While not proprietary commercial data, this approach is standard in academic maritime research and optimization prototypes."

This demonstrates:
- ✅ Technical rigor
- ✅ Transparency
- ✅ Industry-accepted methodology
- ✅ Reproducibility